# Survivor Tutorial

A guide to running Survivor to identify valuable data for model evaluation.

## What is Survivor?

Survivor is a **model evaluation tool** designed to:

1. identify which images in a dataset are most valuable for comparing model performance
2. provide insight into model performance on subsets of the image dataset

### Identifyng most valuable images for model comparison
Comparing various models on a large dataset can be compute intensive and time consuming. If you are comparing various models and either:
- all models perform well on some types of images 
- all models perform poorly on some types of images

then those images aren't useful in differentiating the models.  If we then want to compare some similar models (e.g. models with continued training), and we assume the models will still perform well and poorly respectively on the same types of images as before, then in that case we can remove those images from subsequent model comparisons to speed up the comparison and hone in on the performance differences between the models.

Survivor makes this possible by categorizing images into three groups:
- **Easy** - High agreement, good model performance
- **Hard** - High agreement, low model performance
- **On the Bubble (OtB)** - Low agreement, varying model performance


### Provide model performance insight on subsets of the image dataset
After performing the "Easy", "Hard", or "On the Bubble", categorization, Survivor can help reveal correlations between model performance and image subsets. E.g. all models struggle with images taken at night or underwater.  You could then focus on gathering more of these types of images to use for additional training of your models.

## How does Survivor work?

The Survivor algorithm consists of 4 steps which occur for each image in the dataset:
1. For each model, calculate a performance **metric** describing the performance of the model on the image
2. Calculate a **psuedo-mode** of the model metrics for that image
3. Calculate an **agreement score** among the models (number of metric results close to the mode / total number of models)
4. Classify the image as "Easy", "Hard", or "On the Bubble" using thresholds and/or percentiles

After classification, survivor also provides visualization and analysis tools to analyze the results

### 1. Calculate performance metric

The metric quantifies the quality of model result (bounding boxes, labels, and confidence) compared to the ground truth data.  The default metric is **mean average precision (mAP)** which ranges from 0-1.  Read more about it in the [torchmetrics documentation](https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision.html).

e.g. For a dataset with 4 images run with 5 models.  Your metric values might look like
```python
{
    # 'model_name': [img1_mAP, img2_mAP, etc.]
    'model_1': [0.95, 0.73, 0.9, 0.48],
    'model_2': [0.73, 0.81, 0.8, 0.45],
    'model_3': [0.41, 0.75, 0.86, 0.52],
    'model_4': [0.74, 0.82, 0.83, 0.49],
    'model_5': [0.76, 0.78, 0.88, 0.47]
}
```

### 2. Calculate a pseudo mode of model metrics

The pseudo mode is used to quantify model agreement. A strict mode is often inadequate for quantifying model agreement because it doesn't account for values that are merely close. Therefore, Survivor employs a "pseudo mode" by default.  This method involves moving a fixed-width window across the range of metric values for a specific image. The pseudo mode is then determined by the window position that encompasses the highest concentration (fraction) of metric values.

![pseudo-mode](../assets/survivor-pseudo-mode.png)

For example, for "img1" in the example above the pseudo-mode would be the window position encompassing the 3 closely spaced points (0.73) since they simultaneously fit in the window.

### 3. Calculate the agreement score

The agreement score is calculated as the relative frequency of the modal value.  For example, in the diagram above the agreement score would be 3/5 = `0.6` since 3 of the 5 metric values are in the window at the modal position.

### 4. Classify the image as "Easy", "Hard", or "On the Bubble" using thresholds and/or percentiles

Survivor uses a two-step approach to classify each data point:

1. Data is classified as **On the Bubble** if its agreement score is below a threshold:
   - Either the bottom x% of agreement scores (using `otb_percentile`)
   - Or below an absolute threshold (using `otb_threshold`)

2. Remaining data (with higher agreement) is classified as:
   - **Hard** if the agreed metric score <= `easy_hard_threshold`
   - **Easy** if the agreed metric score > `easy_hard_threshold`

Continuing with the example, and assuming thresholds of 0.5 for both the otb_threshold and the easy_hard_threshold.  The point above would be classified as "Easy" since the agreement score (0.6) is greater than the on the bubble threshold (0.5) and the pseudo-mode (0.73) is greater than the easy_hard_threshold (0.5).

## Running Survivor

In [ ]:
from jatic_ri import cache_path
from jatic_ri.object_detection.datasets import (
    CocoDetectionDataset,
)
from pathlib import Path
from jatic_ri.object_detection.metrics import map50_torch_metric_factory

from pathlib import Path

import numpy as np

from jatic_ri import cache_path
from jatic_ri.object_detection.datasets import (
    CocoDetectionDataset,
)
from jatic_ri.object_detection.models import TorchvisionODModel
from jatic_ri.object_detection.test_stages import (
    SurvivorTestStage,
)

### 2. Load the dataset

In [ ]:
BASE_DIR = Path.cwd().parents[1]

# Example COCO dataset used for testing
root_path = BASE_DIR / "tests/testing_utilities/example_data/coco_dataset"
ann_file_path = BASE_DIR / "tests/testing_utilities/example_data/coco_dataset/ann_file.json"

dataset = CocoDetectionDataset(root=root_path, ann_file=ann_file_path, dataset_id="coco-example")
print(f"Dataset loaded with {len(dataset)} images")

### 3. Create a SurvivorConfig object

The `SurvivorConfig` object controls various aspects of how Survivor processes and categorizes data.  The most common configuration options are:
- **column_names** (ColumnNameConfig): The configuration of the column names of the dataframes used by Survivor.
- **conversion_type** (str): The metric values can be grouped differently.  Available options are "original" (default), "binned", and "rounded". Default: "original"
- **conversion_args** (dict[str, Any]): The arguments relevant to the given `conversion_type` conversion. Default: dict()
- **comparison_window_radius** (float): The radius (in metric_column units) of the window to use when comparing metric scores to find the mode. Can be set to 0 for strict comparison. Default: 0.05.
- **bin_comparison_value** (str): The value within a bin which is compared to the easy/hard threshold. Defaults to the midpoint.
- **otb_percentile** (float): A float between 0 and 1, inclusive. The percentile of agreement scores below which data will be classified as "on the bubble". e.g. "0.2" will select the 20% lowest agreement score pieces of data as on the bubble. Default: 0.2.
- **otb_threshold** (Optional[float]): Optional float between 0 and 1, inclusive. An optional fixed agreement score threshold which overrides the percentile-based classification if passed. If model agreement is below this threshold, the data is considered to be "on the bubble". Default: None.
- **easy_hard_threshold** (float): If models have an agreed metric score above this threshold the data is considered "easy", and if models have an agreed metric score equal to or below this threshold the data is considered "hard". Default: 0.5.
- **heatmap_plot_columns** (list[str]): Metadata columns for which to create resulting heatmaps for each label.  Default: None.

In [ ]:
from jatic_ri._common.test_stages.impls.survivor_test_stage import SurvivorConfig

config = SurvivorConfig(
    metric_column="mAP",
    comparison_window_radius=0.05,
    otb_threshold=0.5,
    heatmap_plot_columns=["average_box_size", "number_of_boxes", "number_of_unique_classes"],
)

### 4. Load models on the selected device

In [ ]:
models_config = {
    "fasterrcnn": "fasterrcnn_resnet50_fpn",
    "retinanet": "retinanet_resnet50_fpn",
}

models = {}
for model_name in models_config:
    models[model_name] = TorchvisionODModel(
        model_name=models_config[model_name],
        model_id=models_config[model_name],
    )
print(f"Loaded models: {', '.join(models_config.keys())}")

### 5. Create and configure Survivor test stage

In [ ]:
test_stage = SurvivorTestStage(
    config=config,
)

test_stage.load_models(models=models)
test_stage.load_dataset(
    dataset=dataset,
    dataset_id="coco",
)

# load metric to compare model performance with ground truth labels
metric_od = map50_torch_metric_factory()
test_stage.load_metric(metric_od, metric_od.return_key)

### 6. Run Survivor

Now we can run the Survivor pipeline to analyze our data.

In [ ]:
%%time
%%capture  
# %%capture necessary when running in jupyterlab in order to suppress image outputs.
# See https://gitlab.jatic.net/jatic/morse/survivor/-/issues/196 for more info.

run = test_stage.run(use_stage_cache=False)

### 7. View Results

#### Dataframe Outputs

In [ ]:
# raw outputs
run.outputs.raw_output_df

In [ ]:
# per image per model input metric with label added
run.outputs.metrics_with_survivor_label_df

#### Metadata heatmaps

To reveal additional insights, heatmaps can help visualize the distribution of data within key metadata columns. For each selected metadata column, values are grouped into distinct categories (buckets). The heatmap then clearly shows the proportion of items labeled "Easy," "Hard," or "On the Bubble" that fall into each of these metadata buckets, allowing for an understanding of how metadata characteristics correlate with these labels.

In [ ]:
for heatmap_plot in run.outputs.heatmap_plots:
    display(heatmap_plot)

## Conclusion

Survivor helps you identify which data points in your dataset are most valuable for model evaluation and comparison. By focusing on "On the Bubble" data, you can:

1. **Reduce evaluation time** by testing on a smaller, more discriminative subset
2. **Improve model comparison** by focusing on data where models actually differ
3. **Find patterns** in what makes some data more challenging than others

For more detailed information, check out the [Survivor documentation](../explanation/how_survivor_works.md).